# 38 - Benchmark JAFFE: LOSO (10-Fold = 10 Subjects)

**Dataset:** JAFFE - 213 gambar, 7 emosi, 10 subjek
**Evaluasi:** LOSO (10 fold)
**Skenario:** B1 only
**Kelas:** 7-class dan 4-class

In [1]:
import sys, os, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from collections import Counter
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, accuracy_score

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from training.models import (
    EmotionCNN, EmotionFCNN, IntermediateFusion,
    EmotionCNNTransfer, IntermediateFusionTransfer,
)
from training.utils import train_model, full_evaluation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

BATCH_SIZE = 16
EPOCHS = 50
PATIENCE = 15
OUTPUT_DIR = PROJECT_ROOT / "models" / "benchmark" / "jaffe_loso"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODELS = [
    ("CNN", EmotionCNN, "cnn", 0.0001),
    ("FCNN", EmotionFCNN, "fcnn", 0.0001),
    ("Intermediate", IntermediateFusion, "fusion", 0.0001),
    ("CNN_TL", EmotionCNNTransfer, "cnn", 0.00005),
    ("Intermediate_TL", IntermediateFusionTransfer, "fusion", 0.00005),
]
print("Setup complete.")

Device: cuda
GPU: Tesla T4
Setup complete.


In [2]:
def make_loader(images, landmarks, labels, model_type, batch_size=16, shuffle=True):
    img_t = torch.from_numpy(images).permute(0, 3, 1, 2)
    lm_t = torch.from_numpy(landmarks)
    y_t = torch.from_numpy(labels).long()
    if model_type == "cnn": ds = TensorDataset(img_t, y_t)
    elif model_type == "fcnn": ds = TensorDataset(lm_t, y_t)
    else: ds = TensorDataset(img_t, lm_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=True)

def train_fold(ModelClass, model_type, lr, train_img, train_lm, train_y,
               test_img, test_lm, test_y, emotions, fold_dir):
    n_val = max(1, int(len(train_y) * 0.15))
    perm = np.random.RandomState(42).permutation(len(train_y))
    val_i, tr_i = perm[:n_val], perm[n_val:]
    tr_l = make_loader(train_img[tr_i], train_lm[tr_i], train_y[tr_i], model_type, BATCH_SIZE)
    vl_l = make_loader(train_img[val_i], train_lm[val_i], train_y[val_i], model_type, BATCH_SIZE, False)
    te_l = make_loader(test_img, test_lm, test_y, model_type, BATCH_SIZE, False)
    model = ModelClass(num_classes=len(emotions)).to(device)
    sp = str(fold_dir / "model.pth")
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=8, min_lr=1e-7)
    train_model(model, tr_l, vl_l, nn.CrossEntropyLoss(), opt, sch, device, model_type, EPOCHS, PATIENCE, sp)
    model.load_state_dict(torch.load(sp, map_location=device, weights_only=True))
    r = full_evaluation(model, te_l, nn.CrossEntropyLoss(), device, model_type, emotions)
    os.remove(sp)
    return {"accuracy": float(r["test_accuracy"]), "macro_f1": float(r["test_macro_f1"]),
            "weighted_f1": float(r["test_weighted_f1"])}

def late_fusion_fold(train_img, train_lm, train_y, test_img, test_lm, test_y, num_classes, fold_dir):
    n_val = max(1, int(len(train_y) * 0.15))
    perm = np.random.RandomState(42).permutation(len(train_y))
    val_i, tr_i = perm[:n_val], perm[n_val:]
    cnn = EmotionCNN(num_classes=num_classes).to(device)
    o1 = torch.optim.Adam(cnn.parameters(), lr=0.0001)
    s1 = torch.optim.lr_scheduler.ReduceLROnPlateau(o1, mode="max", factor=0.5, patience=8, min_lr=1e-7)
    train_model(cnn, make_loader(train_img[tr_i],train_lm[tr_i],train_y[tr_i],"cnn",BATCH_SIZE),
                make_loader(train_img[val_i],train_lm[val_i],train_y[val_i],"cnn",BATCH_SIZE,False),
                nn.CrossEntropyLoss(), o1, s1, device, "cnn", EPOCHS, PATIENCE, str(fold_dir/"cnn.pth"))
    fcnn = EmotionFCNN(num_classes=num_classes).to(device)
    o2 = torch.optim.Adam(fcnn.parameters(), lr=0.0001)
    s2 = torch.optim.lr_scheduler.ReduceLROnPlateau(o2, mode="max", factor=0.5, patience=8, min_lr=1e-7)
    train_model(fcnn, make_loader(train_img[tr_i],train_lm[tr_i],train_y[tr_i],"fcnn",BATCH_SIZE),
                make_loader(train_img[val_i],train_lm[val_i],train_y[val_i],"fcnn",BATCH_SIZE,False),
                nn.CrossEntropyLoss(), o2, s2, device, "fcnn", EPOCHS, PATIENCE, str(fold_dir/"fcnn.pth"))
    cnn.load_state_dict(torch.load(fold_dir/"cnn.pth", map_location=device, weights_only=True))
    fcnn.load_state_dict(torch.load(fold_dir/"fcnn.pth", map_location=device, weights_only=True))
    cnn.eval(); fcnn.eval()
    ti = torch.from_numpy(test_img).permute(0,3,1,2).to(device)
    tl = torch.from_numpy(test_lm).to(device)
    with torch.no_grad():
        cp = torch.softmax(cnn(ti), dim=1).cpu().numpy()
        fp = torch.softmax(fcnn(tl), dim=1).cpu().numpy()
    best_f1, best_w = 0, 0.5
    for w in np.arange(0.0, 1.05, 0.05):
        preds = (w*cp+(1-w)*fp).argmax(axis=1)
        f1 = f1_score(test_y, preds, average="macro", zero_division=0)
        if f1 > best_f1: best_f1=f1; best_w=w; best_preds=preds
    acc = accuracy_score(test_y, best_preds)
    wf1 = f1_score(test_y, best_preds, average="weighted", zero_division=0)
    for f in ["cnn.pth","fcnn.pth"]: (fold_dir/f).unlink(missing_ok=True)
    return {"accuracy": acc, "macro_f1": best_f1, "weighted_f1": wf1}

print("Helper functions ready.")

def run_loso(dataset_name, data_dir, num_classes, emotions):
    print(f"\n{'='*70}")
    print(f"  BENCHMARK: {dataset_name} ({num_classes}-class, LOSO)")
    print(f"{'='*70}")
    images = np.load(data_dir/"X_images.npy"); landmarks = np.load(data_dir/"X_landmarks.npy")
    labels = np.load(data_dir/"y_labels.npy"); subjects = np.load(data_dir/"subjects.npy", allow_pickle=True)
    unique_subjects = sorted(set(subjects))
    subject_indices = {s: np.where(subjects==s)[0] for s in unique_subjects}
    n_folds = len(unique_subjects)
    print(f"  Samples: {len(labels)}, Subjects: {n_folds}")
    all_results = {}
    models_to_run = MODELS + [("Late_Fusion", None, "late", 0.0001)]
    for model_name, ModelClass, model_type, lr in models_to_run:
        key = f"{model_name}_B1"
        print(f"\n  >> {key} ({n_folds} folds)")
        fold_results = []
        model_dir = OUTPUT_DIR / f"{dataset_name}_{num_classes}c" / key
        os.makedirs(model_dir, exist_ok=True)
        for fi, test_subj in enumerate(unique_subjects):
            test_idx = subject_indices[test_subj]
            train_idx = np.concatenate([subject_indices[s] for s in unique_subjects if s != test_subj])
            fold_dir = model_dir / f"fold_{fi}"; os.makedirs(fold_dir, exist_ok=True)
            if model_type == "late":
                r = late_fusion_fold(images[train_idx], landmarks[train_idx], labels[train_idx],
                                     images[test_idx], landmarks[test_idx], labels[test_idx], num_classes, fold_dir)
            else:
                r = train_fold(ModelClass, model_type, lr, images[train_idx], landmarks[train_idx], labels[train_idx],
                               images[test_idx], landmarks[test_idx], labels[test_idx], emotions, fold_dir)
            fold_results.append(r)
            try: fold_dir.rmdir()
            except: pass
        f1s = [r["macro_f1"] for r in fold_results]; accs = [r["accuracy"] for r in fold_results]
        print(f"     F1: {np.mean(f1s):.4f} +/- {np.std(f1s):.4f}  Acc: {np.mean(accs):.4f} +/- {np.std(accs):.4f}")
        all_results[key] = {"model": model_name, "macro_f1_mean": float(np.mean(f1s)),
            "macro_f1_std": float(np.std(f1s)), "accuracy_mean": float(np.mean(accs)),
            "accuracy_std": float(np.std(accs)), "n_folds": n_folds, "per_fold": fold_results}
    save_path = OUTPUT_DIR / f"{dataset_name}_{num_classes}c_loso_results.json"
    with open(save_path, "w") as f: json.dump(all_results, f, indent=2)
    print(f"\n  Saved: {save_path}")
    return all_results

Helper functions ready.


## Run JAFFE LOSO

In [3]:
BENCHMARK_DIR = PROJECT_ROOT / "data" / "benchmark"
EMOTIONS_7 = ["neutral", "happy", "sad", "angry", "fearful", "disgusted", "surprised"]
EMOTIONS_4 = ["neutral", "happy", "sad", "negative"]

res_7c = run_loso("jaffe", BENCHMARK_DIR / "jaffe_7class", 7, EMOTIONS_7)
res_4c = run_loso("jaffe", BENCHMARK_DIR / "jaffe_4class", 4, EMOTIONS_4)


  BENCHMARK: jaffe (7-class, LOSO)


  Samples: 213, Subjects: 10

  >> CNN_B1 (10 folds)


 Epoch  Train Loss  Train Acc   Val Loss   Val Acc   Val F1         LR
---------------------------------------------------------------------------


     1      2.0698     0.1296     1.9448    0.1786   0.0433   0.000100  (1.6s)


     2      2.0198     0.1914     1.9892    0.2143   0.0504   0.000100  (1.2s)


...

    11      1.1138     0.5583     1.1950    0.5357   0.3205   0.000100  (0.0s)
    12      1.1083     0.5521     1.2290    0.5000   0.3385   0.000100  (0.0s)
    13      1.0861     0.5951     1.2396    0.6429   0.4745   0.000100  (0.0s)
    14      1.0824     0.5951     1.2067    0.6786   0.5575   0.000100  (0.0s)
    15      1.1033     0.5644     1.1811    0.6786   0.5749   0.000100  (0.0s)


    16      1.0433     0.5644     1.1894    0.5714   0.4484   0.000100  (0.0s)
    17      1.0692     0.5890     1.1023    0.5714   0.2663   0.000100  (0.0s)
    18      1.0697     0.5583     1.1493    0.6429   0.4600   0.000100  (0.0s)
    19      1.0512     0.5828     1.1982    0.5714   0.3245   0.000100  (0.0s)
    20      1.0627     0.5828     1.0895    0.6429   0.3750   0.000100  (0.0s)


    21      1.0162     0.6135     1.1450    0.6071   0.4342   0.000100  (0.0s)
    22      1.0080     0.6380     1.1517    0.6786   0.5746   0.000100  (0.0s)
    23      1.0333     0.5706     1.1825    0.6071   0.3485   0.000100  (0.0s)
    24      1.0081     0.6196     1.0913    0.7143   0.4300   0.000100  (0.0s)
    25      0.9607     0.6135     1.1555    0.5714   0.4369   0.000050  (0.0s)
    26      1.0181     0.6074     1.1272    0.6071   0.4345   0.000050  (0.0s)


    27      0.9575     0.6564     1.0827    0.6429   0.4595   0.000050  (0.0s)
    28      0.9721     0.6564     1.0650    0.7143   0.5531   0.000050  (0.0s)
    29      0.9917     0.6074     1.1017    0.5714   0.4137   0.000050  (0.0s)
    30      0.9615     0.6258     1.0859    0.6071   0.5330   0.000050  (0.0s)

Early stopping at epoch 30. Best epoch: 15 (val_f1=0.5749)

Best: epoch 15, val_acc=0.6786, val_f1=0.5749
Model saved: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/jaffe_loso/jaffe_4c/Late_Fusion_B1/fold_9/fcnn.pth


     F1: 0.5299 +/- 0.1260  Acc: 0.6637 +/- 0.1150

  Saved: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/jaffe_loso/jaffe_4c_loso_results.json


## Ringkasan JAFFE LOSO

In [4]:
for nc_label, res in [("7-class", res_7c), ("4-class", res_4c)]:
    print(f"\n{'='*70}")
    print(f"  JAFFE {nc_label} - LOSO Results")
    print(f"{'='*70}")
    print(f"  {'Model':<25} {'Macro F1':>18} {'Accuracy':>18}")
    print(f"  {'-'*63}")
    for key in sorted(res.keys(), key=lambda k: -res[k]["macro_f1_mean"]):
        r = res[key]
        print(f"  {key:<25} {r['macro_f1_mean']:.4f} +/- {r['macro_f1_std']:.4f}  {r['accuracy_mean']:.4f} +/- {r['accuracy_std']:.4f}")


  JAFFE 7-class - LOSO Results
  Model                               Macro F1           Accuracy
  ---------------------------------------------------------------
  Late_Fusion_B1            0.4667 +/- 0.0924  0.5396 +/- 0.0840
  CNN_TL_B1                 0.4260 +/- 0.1430  0.4988 +/- 0.1221
  FCNN_B1                   0.3043 +/- 0.1569  0.3824 +/- 0.1577
  Intermediate_TL_B1        0.2925 +/- 0.1556  0.3633 +/- 0.1333
  CNN_B1                    0.2486 +/- 0.1110  0.3534 +/- 0.1055
  Intermediate_B1           0.1294 +/- 0.0703  0.2258 +/- 0.0631

  JAFFE 4-class - LOSO Results
  Model                               Macro F1           Accuracy
  ---------------------------------------------------------------
  Late_Fusion_B1            0.5299 +/- 0.1260  0.6637 +/- 0.1150
  CNN_TL_B1                 0.5096 +/- 0.1551  0.6840 +/- 0.0769
  Intermediate_TL_B1        0.4496 +/- 0.2142  0.6047 +/- 0.1720
  FCNN_B1                   0.4312 +/- 0.1936  0.5402 +/- 0.1323
  CNN_B1              